# 03 특징 엔지니어링

**목적:** 02장에서 만든 주간 `df_weekly.parquet`에서 **2-type(E·C) 66 시계열**에 **예측·분류용 피처**를 추가합니다.

| 피처 유형 | 예시 | 용도 |
|----------|------|------|
| Lag | lag_1~8 | 과거 판매 수준 |
| Rolling | roll_mean/std | 추세·변동성 |
| 간헐성 | zero_ratio_12 | SBC(Lumpy/Intermittent) 포착 |
| 외생·달력 | oil, holiday, month | 수요 설명 변수 |

**산출물:** `df_weekly_features.parquet` (2-type, 15,972행)


### ⓪ 환경 설정 및 주간 데이터 로드

02장 산출물 `df_weekly.parquet`를 읽고, type×family×yearweek 순으로 정렬합니다.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 경로 설정
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))
from utils.paths import DATA_PROCESSED
from utils.config import filter_selected_types, selected_type_list

# 02장 주간 집계 데이터 로드 → 실험 대상 2-type(고변동 E·저변동 C)만 필터
dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
dfw = filter_selected_types(dfw)
dfw = dfw.sort_values(['type', 'family', 'yearweek']).reset_index(drop=True)
print('실험 대상 type:', selected_type_list(), '| 시계열 수:', dfw.groupby(['type','family']).ngroups)
dfw.head()

실험 대상 type: ['E', 'C'] | 시계열 수: 66


,type,family,year,week,yearweek,sales,onpromotion,transactions,dcoilwtico,is_holiday
0,C,AUTOMOTIVE,2013,1,201301,245.0,0,79242.0,93.110000,True
1,C,AUTOMOTIVE,2013,2,201302,291.0,0,99052.0,93.538571,False
2,C,AUTOMOTIVE,2013,3,201303,259.0,0,98994.0,94.927143,False
3,C,AUTOMOTIVE,2013,4,201304,282.0,0,98596.0,95.531429,False
4,C,AUTOMOTIVE,2013,5,201305,338.0,0,102552.0,97.190000,False


### ① Lag·Rolling·간헐성 피처

시계열 그룹(`type`, `family`)별로:
- **Lag:** 1·2·4·8주 전 판매 (shift)
- **Rolling:** 과거 4·8·12주 이동평균·표준편차 (현재 주 제외: `shift(1)`)
- **zero_ratio_12:** 최근 12주 중 판매=0 비율 → 간헐 수요 지표

In [2]:
# --- Lag: N주 전 판매량 (시계열 그룹별 shift) ---
for lag in [1, 2, 4, 8]:
    dfw[f'lag_{lag}'] = dfw.groupby(['type', 'family'])['sales'].shift(lag)

# --- Rolling: 과거 w주 이동평균·표준편차 (현재 주는 제외) ---
for w in [4, 8, 12]:
    dfw[f'roll_mean_{w}'] = dfw.groupby(['type', 'family'])['sales'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean()
    )
    dfw[f'roll_std_{w}'] = dfw.groupby(['type', 'family'])['sales'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).std()
    )

# --- 간헐성: 최근 12주 0판매 비율 (SBC 분류에 활용) ---
dfw['zero_ratio_12'] = dfw.groupby(['type', 'family'])['sales'].transform(
    lambda s: s.shift(1).rolling(12, min_periods=1).apply(lambda x: (x == 0).mean())
)
dfw.head()


,type,family,year,week,yearweek,sales,onpromotion,transactions,dcoilwtico,is_holiday,...,lag_2,lag_4,lag_8,roll_mean_4,roll_std_4,roll_mean_8,roll_std_8,roll_mean_12,roll_std_12,zero_ratio_12
0,C,AUTOMOTIVE,2013,1,201301,245.0,0,79242.0,93.110000,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C,AUTOMOTIVE,2013,2,201302,291.0,0,99052.0,93.538571,False,...,NaN,NaN,NaN,245.00,NaN,245.00,NaN,245.00,NaN,0.0
2,C,AUTOMOTIVE,2013,3,201303,259.0,0,98994.0,94.927143,False,...,245.0,NaN,NaN,268.00,32.526912,268.00,32.526912,268.00,32.526912,0.0
3,C,AUTOMOTIVE,2013,4,201304,282.0,0,98596.0,95.531429,False,...,291.0,NaN,NaN,265.00,23.579652,265.00,23.579652,265.00,23.579652,0.0
4,C,AUTOMOTIVE,2013,5,201305,338.0,0,102552.0,97.190000,False,...,259.0,245.0,NaN,269.25,21.045585,269.25,21.045585,269.25,21.045585,0.0


### ② 달력·연도 피처

`yearweek`(YYYYWW)에서 **월(month)**, **연도(year_feat)** 파생. ML 모델(07장 RF/XGB) 입력용 피처 목록을 확인합니다.

In [3]:
# yearweek(YYYYWW)에서 달력 파생 변수
dfw['month'] = ((dfw['yearweek'] % 100) // 4 + 1).clip(1, 12)  # 주차→월 근사
dfw['year_feat'] = dfw['yearweek'] // 100

# sales를 제외한 피처 컬럼 수 확인
feat_cols = [c for c in dfw.columns if c != 'sales']
print('feature columns:', len(feat_cols))
dfw[feat_cols].head()


feature columns: 22


,type,family,year,week,yearweek,onpromotion,transactions,dcoilwtico,is_holiday,lag_1,...,lag_8,roll_mean_4,roll_std_4,roll_mean_8,roll_std_8,roll_mean_12,roll_std_12,zero_ratio_12,month,year_feat
0,C,AUTOMOTIVE,2013,1,201301,0,79242.0,93.110000,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2013
1,C,AUTOMOTIVE,2013,2,201302,0,99052.0,93.538571,False,245.0,...,NaN,245.00,NaN,245.00,NaN,245.00,NaN,0.0,1,2013
2,C,AUTOMOTIVE,2013,3,201303,0,98994.0,94.927143,False,291.0,...,NaN,268.00,32.526912,268.00,32.526912,268.00,32.526912,0.0,1,2013
3,C,AUTOMOTIVE,2013,4,201304,0,98596.0,95.531429,False,259.0,...,NaN,265.00,23.579652,265.00,23.579652,265.00,23.579652,0.0,2,2013
4,C,AUTOMOTIVE,2013,5,201305,0,102552.0,97.190000,False,282.0,...,NaN,269.25,21.045585,269.25,21.045585,269.25,21.045585,0.0,2,2013


### ③ 피처 포함 주간 테이블 저장

`sales`를 타깃으로, 나머지 22개 컬럼을 피처로 `df_weekly_features.parquet`에 저장합니다.

In [4]:
# 피처 포함 주간 테이블 저장 (07장 ML 예측 입력)
out = DATA_PROCESSED / 'df_weekly_features.parquet'
dfw.to_parquet(out, index=False)
print('저장:', out, '| shape:', dfw.shape)


저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\df_weekly_features.parquet | shape: (15972, 23)


## 분석 요약

### 생성 피처 (총 22개, sales 제외)
| 구분 | 피처 |
|------|------|
| **Lag** | `lag_1`, `lag_2`, `lag_4`, `lag_8` (1~8주 전 판매) |
| **Rolling** | `roll_mean/std_{4,8,12}` (과거 이동평균·표준편차) |
| **간헐성** | `zero_ratio_12` (최근 12주 0판매 비율) |
| **외생·달력** | `onpromotion`, `transactions`, `dcoilwtico`, `is_holiday`, `month`, `year_feat` |

### 데이터 형태
- **15,972행 × 23컬럼** (66 시계열 × 242주)
- lag 피처는 시계열 첫 주에 결측 66건(시리즈당 1건) — 정상적인 shift 결과

### 활용
- `07` ML 예측(RF, XGBoost)의 패널 학습 피처로 사용
- rolling·zero_ratio는 **간헐 수요(Lumpy/Intermittent)** 패턴 포착에 유리